# Problem 2: Transformer components from scratch

Run the cells in order with Restart and Run All.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)


In [2]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout=0.0):
    scale = q.size(-1) ** -0.5
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = torch.softmax(scores, dim=-1)
    if dropout > 0:
        weights = F.dropout(weights, p=dropout, training=True)
    out = torch.matmul(weights, v)
    return out, weights

def causal_mask(length):
    return torch.tril(torch.ones(length, length, dtype=torch.bool))

q = torch.randn(2, 4, 8)
k = torch.randn(2, 4, 8)
v = torch.randn(2, 4, 8)
out, weights = scaled_dot_product_attention(q, k, v)
assert out.shape == q.shape
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=1e-5)
print("Attention output shape:", tuple(out.shape))


Attention output shape: (2, 4, 8)


In [3]:
q = torch.tensor([[[1.0, 0.0], [0.5, 0.5], [0.0, 1.0]]])
k = torch.tensor([[[1.0, 0.0], [0.2, 0.8], [0.9, 0.1]]])
v = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]])
out, weights = scaled_dot_product_attention(q, k, v)
ref = F.scaled_dot_product_attention(q, k, v)
assert torch.allclose(out, ref, atol=1e-5, rtol=1e-5)
mask = causal_mask(3)
q2 = torch.randn(1, 3, 2)
k2 = torch.randn(1, 3, 2)
v2 = torch.randn(1, 3, 2)
out2, weights2 = scaled_dot_product_attention(q2, k2, v2, mask=mask.unsqueeze(0).unsqueeze(0))
assert torch.allclose(weights2 * (~mask.unsqueeze(0).unsqueeze(0)), torch.zeros_like(weights2))
print("Causal mask and PyTorch match ok")


Causal mask and PyTorch match ok


In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        q = self.W_q(x).view(x.size(0), x.size(1), self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(x.size(0), x.size(1), self.n_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(x.size(0), x.size(1), self.n_heads, self.d_head).transpose(1, 2)
        attn_out, _ = scaled_dot_product_attention(q, k, v, mask=mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(x.size(0), x.size(1), self.d_model)
        return self.W_o(attn_out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = np.arange(max_len)[:, None]
        div = np.power(10000, np.arange(0, d_model, 2) / d_model)
        pe = np.zeros((max_len, d_model), dtype=np.float32)
        pe[:, 0::2] = np.sin(pos / div)
        pe[:, 1::2] = np.cos(pos / div)
        self.register_buffer("pe", torch.tensor(pe).unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]

class EncoderBlock(nn.Module):
    def __init__(self, d_model, d_ff=512, n_heads=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        residual = x
        x = self.ln1(x)
        x = self.mha(x, mask)
        x = self.dropout(x)
        x = residual + x
        residual = x
        x = self.ln2(x)
        x = self.ff(x)
        x = self.dropout(x)
        return residual + x

block = EncoderBlock(8, 16, n_heads=2)
x = torch.randn(2, 4, 8)
y = block(x)
assert y.shape == x.shape
print("Encoder block output shape:", tuple(y.shape))


Encoder block output shape: (2, 4, 8)


In [5]:
def run_problem2_smoke_tests():
    q = torch.randn(2, 4, 8)
    k = torch.randn(2, 4, 8)
    v = torch.randn(2, 4, 8)
    out, weights = scaled_dot_product_attention(q, k, v)
    assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=1e-5)
    block = EncoderBlock(8, 16, n_heads=2)
    x = torch.randn(2, 4, 8)
    out_block = block(x)
    assert out_block.shape == x.shape
    return {"attention_shape": tuple(out.shape), "block_shape": tuple(out_block.shape)}

print(run_problem2_smoke_tests())


{'attention_shape': (2, 4, 8), 'block_shape': (2, 4, 8)}
